## Flow Cytometry Analysis — Extra Phenotypes
## PARP / γH2AX Cell-Cycle Distribution
### Author: Eleni Aretaki
### Date: 2026-09-11

### Purpose

This notebook analyzes cell-cycle distributions of PARP-positive and
γH2AX-positive populations across SWI/SNF knockout cell lines.

Data from two independent 48h flow-cytometry experiments are analyzed.

### Analysis workflow

1. Load and annotate PARP and γH2AX cell-cycle datasets
2. Apply predefined quality-control exclusions
3. Remove inactive treatments
4. Generate summary cell-count tables
5. Compare cell-cycle distributions statistically using Mann–Whitney U-tests
6. Correct multiple comparisons using Benjamini–Hochberg FDR
7. Generate stacked barplots showing mean ± SEM
8. Export statistical results and figures

### Statistical comparisons

Two types of comparisons are performed:

- **Genotype effect:** KO vs WT under DMSO
- **Treatment effect:** drug vs matched DMSO within each cell line,
  experiment and DMSO concentration

### Statistical test

Two-sided Mann–Whitney U test with Benjamini–Hochberg FDR correction.

Significance:
- * FDR < 0.05
- ** FDR < 0.01
- *** FDR < 0.001

In [2]:
# -------------------------------
# Imports
# -------------------------------
import glob
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

In [3]:
# -------------------------------
# Analysis configuration
# -------------------------------

PHASES = ["G1", "S", "G2"]

BAD_TREATMENTS = ["Aph", "NIR"]

# Wells excluded based on predefined QC criteria
BAD_WELLS = {
    "Exp1": {
        "Rep1": ["E8"],
        "Rep2": ["B3", "B4"],
        "Rep3": ["E12", "F12"]
    },
    "Exp2": {
        "Rep1": ["A8"],
        "Rep2": ["A8"],
        "Rep3": ["A8"],
        "Rep5": ["A8", "B7", "C12"]
    }
}

# ------------------------------------------------------------
# Additional experiment-specific exclusions
#
# Each dictionary specifies:
# marker, treatment and experiment.
#
# This is kept separate from BAD_WELLS because these are
# experiment/condition-level exclusions rather than individual
# well exclusions.
# ------------------------------------------------------------

ADDITIONAL_EXCLUSIONS = [
    {
        "marker": "yH2AX",
        "treatment": "MMS",
        "Experiment": "Exp2"
    }
]

# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------

OUTPUT_TABLE_DIR = "../Extra_Phenotypes"

OUTPUT_FIGURE_DIR = "../Extra_Phenotypes/Figures"

OUTPUT_STATS_DIR = "../Extra_Phenotypes/Statistics"

os.makedirs(OUTPUT_TABLE_DIR, exist_ok=True)
os.makedirs(OUTPUT_FIGURE_DIR, exist_ok=True)
os.makedirs(OUTPUT_STATS_DIR, exist_ok=True)

In [5]:
# -------------------------------
# Loading data
# -------------------------------
def load_marker_data(folder, marker_keyword, annotation_df, experiment_name):
    """
    Loads all replicate CSVs for a given marker (e.g. PARP, yH2AX)
    from a folder and returns annotated dataframe.
    """

    # find all relevant files
    pattern = os.path.join(folder, f"*{marker_keyword}*cellcycle.csv")
    files = glob.glob(pattern)

    dfs = []

    for file in files:
        df = pd.read_csv(file)

        # Extract replicate number from filename
        rep = os.path.basename(file).split("_")[1]  # Replicate_X
        df["Replicate"] = rep

        # Identify positive vs negative
        df["Population"] = "neg" if "neg" in file else "pos"

        # Add experiment label
        df["Experiment"] = experiment_name

        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True)

    # Extract well ID
    combined["well_ID"] = combined["Sample"].str.extract(r"_([A-H]\d{1,2})_")

    # Merge annotation
    combined = combined.merge(annotation_df, on="well_ID", how="left")

    return combined

In [ ]:
# ------------------------------------------------------------
# Load annotation tables
# ------------------------------------------------------------

annotation1 = pd.read_csv(
    "../Extra_Phenotypes/Exp1/annotation_df.csv",
    encoding="unicode_escape"
)

annotation2 = pd.read_csv(
    "../Extra_Phenotypes/Exp2/annotation_df2.csv",
    encoding="unicode_escape"
)

# ------------------------------------------------------------
# Load PARP
# ------------------------------------------------------------

exp1_parp = load_marker_data(
    folder="../Extra_Phenotypes/Exp1",
    marker_keyword="PARP",
    annotation_df=annotation1,
    experiment_name="Exp1"
)

exp2_parp = load_marker_data(
    folder="../Extra_Phenotypes/Exp2",
    marker_keyword="PARP",
    annotation_df=annotation2,
    experiment_name="Exp2"
)

parp_data = pd.concat(
    [exp1_parp, exp2_parp],
    ignore_index=True
)

# ------------------------------------------------------------
# Load γH2AX
# ------------------------------------------------------------

exp1_yh2ax = load_marker_data(
    folder="../Extra_Phenotypes/Exp1",
    marker_keyword="yH2AX",
    annotation_df=annotation1,
    experiment_name="Exp1"
)

exp2_yh2ax = load_marker_data(
    folder="../Extra_Phenotypes/Exp2",
    marker_keyword="yH2AX",
    annotation_df=annotation2,
    experiment_name="Exp2"
)

yh2ax_data = pd.concat(
    [exp1_yh2ax, exp2_yh2ax],
    ignore_index=True
)

## Quality control and exclusions

The following wells were excluded based on predefined quality-control criteria.
Inactive treatments were also removed.

In [12]:
# ============================================================
# Quality control / exclusions
# ============================================================
def remove_bad_wells(df, bad_dict):
    """
    bad_dict format:
    {
        "Exp1": {
            "Rep1": ["A1", "B2"]
        }
    }
    """
    df_clean = df.copy()

    for exp, rep_dict in bad_dict.items():
        for rep, wells in rep_dict.items():
            
            df_clean = df_clean[
                ~(
                    (df_clean["Experiment"] == exp) &
                    (df_clean["Replicate"] == rep) &
                    (df_clean["well_ID"].isin(wells))
                )
            ]

    return df_clean

def remove_bad_treatments(df, bad_treatments):
    """
    Remove treatments that should not be included in the analysis.
    """

    return df[
        ~df["treatment"].isin(bad_treatments)
    ].copy()

def apply_additional_exclusions(
    df,
    marker_name,
    exclusions
):
    """
    Apply experiment/condition-specific exclusions.

    Each exclusion should contain:

        marker
        treatment
        Experiment
    """

    df_clean = df.copy()

    for exclusion in exclusions:

        if exclusion["marker"] != marker_name:
            continue

        mask = (
            (df_clean["treatment"] == exclusion["treatment"]) &
            (df_clean["Experiment"] == exclusion["Experiment"])
        )

        df_clean = df_clean.loc[~mask].copy()

    return df_clean

In [10]:
# ------------------------------------------------------------
# Apply standard QC
# ------------------------------------------------------------

clean_parp = remove_bad_wells(
    parp_data,
    BAD_WELLS
)

clean_yh2ax = remove_bad_wells(
    yh2ax_data,
    BAD_WELLS
)


clean_parp = remove_bad_treatments(
    clean_parp,
    BAD_TREATMENTS
)

clean_yh2ax = remove_bad_treatments(
    clean_yh2ax,
    BAD_TREATMENTS
)

In [11]:
# ------------------------------------------------------------
# Apply experiment-specific exclusions
# ------------------------------------------------------------

clean_parp = apply_additional_exclusions(
    clean_parp,
    marker_name="PARP",
    exclusions=ADDITIONAL_EXCLUSIONS
)

clean_yh2ax = apply_additional_exclusions(
    clean_yh2ax,
    marker_name="yH2AX",
    exclusions=ADDITIONAL_EXCLUSIONS
)

## Cell-count summary table

Positive and negative populations are merged to generate a single
well-level summary table for PARP and γH2AX.

In [14]:
# -------------------------------
# Create cell number summary table
# -------------------------------

def extract_marker_counts(df, marker):

    """
    Extracts positive and negative cell counts for one marker
    and reshapes into one row per well.

    Output:
    Experiment | Replicate | well_ID | cell_line | treatment |
    marker_pos | marker_neg
    """

    # Define total column names
    if marker == "yH2AX":
        pos_col = "yH2AX_pos_total"
        neg_col = "yH2AX_neg_total"

    elif marker == "PARP":
        pos_col = "PARP_pos_total"
        neg_col = "PARP_neg_total"

    else:
        raise ValueError("Marker must be PARP or yH2AX")


    # Split positive and negative populations
    pos = df[df["Population"] == "pos"].copy()
    neg = df[df["Population"] == "neg"].copy()


    # Select relevant columns
    pos = pos[
        [
            "Experiment",
            "Replicate",
            "well_ID",
            "modification",
            "treatment",
            pos_col
        ]
    ]

    neg = neg[
        [
            "Experiment",
            "Replicate",
            "well_ID",
            "modification",
            "treatment",
            neg_col
        ]
    ]


    # Merge positive and negative wells
    merged = pos.merge(
        neg,
        on=[
            "Experiment",
            "Replicate",
            "well_ID",
            "modification",
            "treatment"
        ],
        how="outer"
    )


    # Rename columns
    merged = merged.rename(
        columns={
            "modification": "cell_line",
            pos_col: f"{marker}_pos",
            neg_col: f"{marker}_neg"
        }
    )


    return merged



# -------------------------------
# Extract PARP and yH2AX counts
# -------------------------------

yh2ax_counts = extract_marker_counts(
    clean_yh2ax,
    "yH2AX"
)

parp_counts = extract_marker_counts(
    clean_parp,
    "PARP"
)


# -------------------------------
# Merge both markers together
# -------------------------------

cell_count_table = yh2ax_counts.merge(
    parp_counts,
    on=[
        "Experiment",
        "Replicate",
        "well_ID",
        "cell_line",
        "treatment"
    ],
    how="outer"
)


# -------------------------------
# Clean cell line names
# -------------------------------

cell_count_table["cell_line"] = (
    cell_count_table["cell_line"]
    .str.replace("C631 + BRM014", "WT+BRM014")
    .str.replace("C631", "WT")
    .str.replace(" KO", "")
)


# -------------------------------
# Sort table
# -------------------------------

cell_count_table = cell_count_table[
    [
        "Experiment",
        "Replicate",
        "well_ID",
        "cell_line",
        "treatment",
        "yH2AX_pos",
        "yH2AX_neg",
        "PARP_pos",
        "PARP_neg"
    ]
].sort_values(
    [
        "Experiment",
        "Replicate",
        "cell_line",
        "treatment"
    ]
)

# -------------------------------
# Save table
# -------------------------------

cell_count_table.to_csv(
    "../Extra_Phenotypes/cell_count_summary_table.csv",
    index=False
)

In [16]:
# ============================================================
# Statistical helper functions
# ============================================================

# ---------------------------------------
# Significance stars
# ---------------------------------------

def get_significance_stars(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""
        
# ---------------------------------------
# Statistics
# ---------------------------------------
def compare_cell_cycle_phases(
    reference_df,
    comparison_df,
    reference_name,
    comparison_name,
    phases=PHASES
):
    """
    Perform Mann–Whitney U tests for G1, S and G2.

    Multiple testing correction is performed across the
    three cell-cycle phases using Benjamini–Hochberg FDR.

    Returns
    -------
    pandas.DataFrame
    """

    results = []

    raw_pvalues = []

    valid_phases = []

    for phase in phases:

        reference_values = (
            reference_df[phase]
            .dropna()
        )

        comparison_values = (
            comparison_df[phase]
            .dropna()
        )

        # ----------------------------------------------------
        # Skip comparison if one group is empty
        # ----------------------------------------------------

        if (
            len(reference_values) == 0 or
            len(comparison_values) == 0
        ):

            results.append({
                "Reference": reference_name,
                "Comparison": comparison_name,
                "Phase": phase,
                "Reference_n": len(reference_values),
                "Comparison_n": len(comparison_values),
                "U_statistic": np.nan,
                "raw_p": np.nan,
                "FDR": np.nan,
                "Significance": ""
            })

            continue

        # ----------------------------------------------------
        # Mann–Whitney U
        # ----------------------------------------------------

        statistic, pvalue = mannwhitneyu(
            reference_values,
            comparison_values,
            alternative="two-sided"
        )

        raw_pvalues.append(pvalue)
        valid_phases.append(phase)

        results.append({
            "Reference": reference_name,
            "Comparison": comparison_name,
            "Phase": phase,
            "Reference_n": len(reference_values),
            "Comparison_n": len(comparison_values),
            "U_statistic": statistic,
            "raw_p": pvalue,
            "FDR": np.nan,
            "Significance": ""
        })

    # --------------------------------------------------------
    # FDR correction
    # --------------------------------------------------------

    if len(raw_pvalues) > 0:

        _, fdr_values, _, _ = multipletests(
            raw_pvalues,
            method="fdr_bh"
        )

        fdr_dict = dict(
            zip(valid_phases, fdr_values)
        )

        for result in results:

            phase = result["Phase"]

            if phase in fdr_dict:

                result["FDR"] = fdr_dict[phase]

                result["Significance"] = (
                    get_significance_stars(
                        fdr_dict[phase]
                    )
                )

    return pd.DataFrame(results)

In [15]:
# ============================================================
# Genotype statistics — DMSO
# ============================================================

def calculate_genotype_statistics(
    df,
    marker_name,
    cell_lines,
    treatment="DMSO"
):
    """
    Compare each specified cell line against the first cell line.

    The first cell line is treated as the WT/reference.

    IMPORTANT:
    DMSO samples from Exp1 and Exp2 are pooled because the same
    genotype comparison is present in both experiments.

    Statistics are calculated separately for each KO vs WT
    comparison.

    Multiple testing correction:
        G1, S and G2 are FDR-corrected within each comparison.
    """

    if len(cell_lines) < 2:
        raise ValueError(
            "At least two cell lines are required."
        )

    wt_name = cell_lines[0]

    # --------------------------------------------------------
    # Positive population + requested treatment
    # --------------------------------------------------------

    analysis_df = df[
        (df["Population"] == "pos") &
        (df["treatment"] == treatment) &
        (df["modification"].isin(cell_lines))
    ].copy()

    all_results = []

    # --------------------------------------------------------
    # Compare every KO against WT
    # --------------------------------------------------------

    for comparison_name in cell_lines[1:]:

        wt_df = analysis_df[
            analysis_df["modification"] == wt_name
        ]

        ko_df = analysis_df[
            analysis_df["modification"] == comparison_name
        ]

        if wt_df.empty or ko_df.empty:

            print(
                f"Skipping {wt_name} vs {comparison_name}: "
                "one group is empty."
            )

            continue

        print("\n" + "=" * 60)
        print(
            f"{marker_name}: "
            f"{wt_name} vs {comparison_name}"
        )
        print(f"Treatment: {treatment}")
        print("Experiments: pooled")
        print("=" * 60)

        results = compare_cell_cycle_phases(
            reference_df=wt_df,
            comparison_df=ko_df,
            reference_name=wt_name,
            comparison_name=comparison_name,
            phases=PHASES
        )

        results.insert(
            0,
            "Analysis",
            "Genotype"
        )

        results.insert(
            1,
            "Marker",
            marker_name
        )

        results.insert(
            2,
            "Treatment",
            treatment
        )

        results.insert(
            3,
            "Experiment",
            "Pooled"
        )

        all_results.append(results)

        # ----------------------------------------------------
        # Print results
        # ----------------------------------------------------

        for _, row in results.iterrows():

            print(
                f"{row['Phase']}: "
                f"n={row['Reference_n']} vs "
                f"{row['Comparison_n']}, "
                f"raw p={row['raw_p']:.4g}, "
                f"FDR={row['FDR']:.4g}, "
                f"{row['Significance']}"
            )

    if len(all_results) == 0:
        return pd.DataFrame()

    return pd.concat(
        all_results,
        ignore_index=True
    )

In [17]:
# ============================================================
# Treatment vs DMSO statistics
# ============================================================

def values_match(a, b):
    """
    Compare two values while treating NaN == NaN.
    Useful for matching DMSO concentrations.
    """

    if pd.isna(a) and pd.isna(b):
        return True

    return a == b


def calculate_treatment_statistics(
    df,
    marker_name,
    cell_lines=None,
    treatments=None
):
    """
    Compare treatment vs matched DMSO.

    For each:
        cell line
        treatment
        experiment
        DMSO concentration

    the treatment is compared with the corresponding DMSO.

    Experiment is therefore explicitly respected for treatment
    comparisons.

    G1, S and G2 p-values are corrected using Benjamini–Hochberg FDR
    separately for each treatment/experiment/DMSO combination.
    """

    if cell_lines is None:

        cell_lines = sorted(
            df["modification"]
            .dropna()
            .unique()
        )

    if treatments is None:

        treatments = sorted(
            [
                treatment
                for treatment in df["treatment"].dropna().unique()
                if treatment != "DMSO"
            ]
        )

    # --------------------------------------------------------
    # Positive population
    # --------------------------------------------------------

    analysis_df = df[
        df["Population"] == "pos"
    ].copy()

    all_results = []

    # --------------------------------------------------------
    # Loop through cell lines
    # --------------------------------------------------------

    for cell_line in cell_lines:

        df_cl = analysis_df[
            analysis_df["modification"] == cell_line
        ].copy()

        if df_cl.empty:

            print(
                f"Skipping {cell_line}: no data."
            )

            continue

        # ----------------------------------------------------
        # Loop through treatments
        # ----------------------------------------------------

        for treatment in treatments:

            drug_df = df_cl[
                df_cl["treatment"] == treatment
            ].copy()

            if drug_df.empty:

                print(
                    f"Skipping {cell_line} - {treatment}: "
                    "no treatment data."
                )

                continue

            # ------------------------------------------------
            # Find unique experiment/DMSO concentration
            # combinations in the treatment dataset
            # ------------------------------------------------

            combinations = (
                drug_df[
                    [
                        "Experiment",
                        "DMSO_conc"
                    ]
                ]
                .drop_duplicates()
            )

            # ------------------------------------------------
            # Analyze every experiment separately
            # ------------------------------------------------

            for _, combination in combinations.iterrows():

                experiment = combination["Experiment"]
                dmso_conc = combination["DMSO_conc"]

                # --------------------------------------------
                # Treatment
                # --------------------------------------------

                treat = drug_df[
                    (drug_df["Experiment"] == experiment) &
                    (
                        drug_df["DMSO_conc"].apply(
                            lambda x: values_match(
                                x,
                                dmso_conc
                            )
                        )
                    )
                ].copy()

                # --------------------------------------------
                # Matching DMSO
                # --------------------------------------------

                dmso_candidates = df_cl[
                    (df_cl["treatment"] == "DMSO") &
                    (df_cl["Experiment"] == experiment)
                ].copy()

                dmso = dmso_candidates[
                    dmso_candidates["DMSO_conc"].apply(
                        lambda x: values_match(
                            x,
                            dmso_conc
                        )
                    )
                ].copy()

                if treat.empty or dmso.empty:

                    print(
                        f"Skipping {cell_line} - {treatment} "
                        f"(Experiment={experiment}, "
                        f"DMSO={dmso_conc}): "
                        "matching data not found."
                    )

                    continue

                print("\n" + "=" * 60)
                print(f"{marker_name}: {cell_line}")
                print(
                    f"DMSO vs {treatment}"
                )
                print(
                    f"Experiment: {experiment}"
                )
                print(
                    f"DMSO concentration: {dmso_conc}"
                )
                print("=" * 60)

                # --------------------------------------------
                # Statistics
                # --------------------------------------------

                results = compare_cell_cycle_phases(
                    reference_df=dmso,
                    comparison_df=treat,
                    reference_name="DMSO",
                    comparison_name=treatment,
                    phases=PHASES
                )

                results.insert(
                    0,
                    "Analysis",
                    "Treatment_vs_DMSO"
                )

                results.insert(
                    1,
                    "Marker",
                    marker_name
                )

                results.insert(
                    2,
                    "Cell_line",
                    cell_line
                )

                results.insert(
                    3,
                    "Treatment",
                    treatment
                )

                results.insert(
                    4,
                    "Experiment",
                    experiment
                )

                results.insert(
                    5,
                    "DMSO_conc",
                    dmso_conc
                )

                all_results.append(results)

                # --------------------------------------------
                # Print results
                # --------------------------------------------

                for _, row in results.iterrows():

                    print(
                        f"{row['Phase']}: "
                        f"n={row['Reference_n']} vs "
                        f"{row['Comparison_n']}, "
                        f"raw p={row['raw_p']:.4g}, "
                        f"FDR={row['FDR']:.4g}, "
                        f"{row['Significance']}"
                    )

    if len(all_results) == 0:
        return pd.DataFrame()

    return pd.concat(
        all_results,
        ignore_index=True
    )

In [19]:
# ============================================================
# Calculate all statistics
# ============================================================

# ------------------------------------------------------------
# PARP — WT vs KO under DMSO
# ------------------------------------------------------------

parp_genotype_stats = calculate_genotype_statistics(
    df=clean_parp,
    marker_name="PARP",
    cell_lines=[
        "C631",
        "ARID1A KO",
        "ARID1B KO",
        "SMARCC1 KO",
        "SMARCA4 KO"
    ],
    treatment="DMSO"
)


# ------------------------------------------------------------
# γH2AX — WT vs KO under DMSO
# ------------------------------------------------------------

yh2ax_genotype_stats = calculate_genotype_statistics(
    df=clean_yh2ax,
    marker_name="yH2AX",
    cell_lines=[
        "C631",
        "ARID1A KO",
        "ARID1B KO",
        "SMARCA4 KO"
    ],
    treatment="DMSO"
)


# ------------------------------------------------------------
# PARP — treatment vs DMSO
#
# Change cell_lines and treatments here whenever you want
# to analyze a different subset.
# ------------------------------------------------------------

parp_treatment_stats = calculate_treatment_statistics(
    df=clean_parp,
    marker_name="PARP",
    cell_lines=[
        "C631",
        "ARID1A KO",
        "ARID1B KO",
        "SMARCC1 KO",
        "SMARCA4 KO"
    ],
    treatments=[
        "MMS"
    ]
)


# ------------------------------------------------------------
# γH2AX — treatment vs DMSO
# ------------------------------------------------------------

yh2ax_treatment_stats = calculate_treatment_statistics(
    df=clean_yh2ax,
    marker_name="yH2AX",
    cell_lines=[
        "C631",
        "ARID1A KO",
        "SMARCA4 KO"
    ],
    treatments=[
        "MMS"
    ]
)


PARP: C631 vs ARID1A KO
Treatment: DMSO
Experiments: pooled
G1: n=12 vs 12, raw p=0.7075, FDR=0.7075, 
S: n=12 vs 12, raw p=0.00355, FDR=0.01065, *
G2: n=12 vs 12, raw p=0.01019, FDR=0.01529, *

PARP: C631 vs ARID1B KO
Treatment: DMSO
Experiments: pooled
G1: n=12 vs 12, raw p=0.002437, FDR=0.003655, **
S: n=12 vs 12, raw p=0.0001962, FDR=0.0005885, ***
G2: n=12 vs 12, raw p=0.3408, FDR=0.3408, 

PARP: C631 vs SMARCC1 KO
Treatment: DMSO
Experiments: pooled
G1: n=12 vs 8, raw p=0.02013, FDR=0.0604, 
S: n=12 vs 8, raw p=0.3054, FDR=0.4581, 
G2: n=12 vs 8, raw p=0.7921, FDR=0.7921, 

PARP: C631 vs SMARCA4 KO
Treatment: DMSO
Experiments: pooled
G1: n=12 vs 12, raw p=0.4705, FDR=0.4705, 
S: n=12 vs 12, raw p=0.2855, FDR=0.4282, 
G2: n=12 vs 12, raw p=0.2252, FDR=0.4282, 

yH2AX: C631 vs ARID1A KO
Treatment: DMSO
Experiments: pooled
G1: n=12 vs 12, raw p=0.002431, FDR=0.005325, **
S: n=12 vs 12, raw p=0.00355, FDR=0.005325, **
G2: n=12 vs 12, raw p=0.5067, FDR=0.5067, 

yH2AX: C631 vs ARID1B

In [20]:
# ============================================================
# Save statistical results
# ============================================================

if not parp_genotype_stats.empty:

    parp_genotype_stats.to_csv(
        os.path.join(
            OUTPUT_STATS_DIR,
            "PARP_genotype_statistics.csv"
        ),
        index=False
    )


if not yh2ax_genotype_stats.empty:

    yh2ax_genotype_stats.to_csv(
        os.path.join(
            OUTPUT_STATS_DIR,
            "yH2AX_genotype_statistics.csv"
        ),
        index=False
    )


if not parp_treatment_stats.empty:

    parp_treatment_stats.to_csv(
        os.path.join(
            OUTPUT_STATS_DIR,
            "PARP_treatment_statistics.csv"
        ),
        index=False
    )


if not yh2ax_treatment_stats.empty:

    yh2ax_treatment_stats.to_csv(
        os.path.join(
            OUTPUT_STATS_DIR,
            "yH2AX_treatment_statistics.csv"
        ),
        index=False
    )


# ------------------------------------------------------------
# Combined statistics table
# ------------------------------------------------------------

all_statistics = pd.concat(
    [
        parp_genotype_stats,
        yh2ax_genotype_stats,
        parp_treatment_stats,
        yh2ax_treatment_stats
    ],
    ignore_index=True
)

all_statistics.to_csv(
    os.path.join(
        OUTPUT_STATS_DIR,
        "Extra_Phenotypes_all_statistics.csv"
    ),
    index=False
)

In [21]:
# ============================================================
# Plotting helper functions
# ============================================================

def calculate_means_and_sem(
    df,
    cell_lines
):
    """
    Calculate mean and SEM for G1, S and G2.
    """

    means = []
    errors = []

    for cell_line in cell_lines:

        sub = df[
            df["modification"] == cell_line
        ]

        means.append(
            [
                sub["G1"].mean(),
                sub["S"].mean(),
                sub["G2"].mean()
            ]
        )

        errors.append(
            [
                sub["G1"].sem(),
                sub["S"].sem(),
                sub["G2"].sem()
            ]
        )

    return (
        np.array(means),
        np.array(errors)
    )


def sanitize_filename(text):
    """
    Make a string safe to use as part of a filename.
    """

    text = str(text)

    invalid_characters = [
        "/",
        "\\",
        ":",
        "*",
        "?",
        '"',
        "<",
        ">",
        "|"
    ]

    for character in invalid_characters:
        text = text.replace(
            character,
            "_"
        )

    return text

In [22]:
# ============================================================
# Plot genotype comparison
# ============================================================

def plot_genotype_comparison(
    df,
    marker_name="PARP",
    cell_lines=None,
    treatment="DMSO",
    labels=None,
    stats=None,
    output=None
):
    """
    Plot cell-cycle distribution for multiple cell lines.

    The first cell line is the reference/WT.

    Significance values are taken from the separately calculated
    statistics dataframe.
    """

    plt.rcParams["font.family"] = "Arial"

    if cell_lines is None:

        raise ValueError(
            "Please specify cell_lines."
        )

    if len(cell_lines) < 2:

        raise ValueError(
            "At least two cell lines are required."
        )

    if labels is None:

        labels = cell_lines

    if len(labels) != len(cell_lines):

        raise ValueError(
            "labels and cell_lines must have the same length."
        )

    if output is None:

        output = (
            f"{marker_name}_"
            f"{treatment}_"
            f"genotype_comparison.pdf"
        )

    # --------------------------------------------------------
    # Filter
    # --------------------------------------------------------

    plot_df = df[
        (df["Population"] == "pos") &
        (df["treatment"] == treatment) &
        (df["modification"].isin(cell_lines))
    ].copy()

    if plot_df.empty:

        print(
            f"No data available for "
            f"{marker_name}, {treatment}."
        )

        return

    # --------------------------------------------------------
    # Means and SEM
    # --------------------------------------------------------

    means, errors = calculate_means_and_sem(
        plot_df,
        cell_lines
    )

    # --------------------------------------------------------
    # Figure size
    # --------------------------------------------------------

    bar_width = 1.0
    min_width = 4
    height = 5

    fig_width = max(
        min_width,
        len(cell_lines) * bar_width
    )

    fig, ax = plt.subplots(
        figsize=(fig_width, height)
    )

    x = np.arange(
        len(cell_lines)
    )

    bottom = np.zeros(
        len(cell_lines)
    )

    colors = [
        "steelblue",
        "lightblue",
        "thistle"
    ]

    # --------------------------------------------------------
    # Plot cell-cycle phases
    # --------------------------------------------------------

    for i, phase in enumerate(PHASES):

        ax.bar(
            x,
            means[:, i],
            bottom=bottom,
            yerr=errors[:, i],
            capsize=1.5,
            width=0.6,
            color=colors[i],
            label=phase,
            error_kw={
                "elinewidth": 0.6,
                "capthick": 0.6
            }
        )

        # ----------------------------------------------------
        # Significance
        # ----------------------------------------------------

        if stats is not None and not stats.empty:

            for j, cell_line in enumerate(
                cell_lines[1:],
                start=1
            ):

                comparison_stats = stats[
                    (stats["Comparison"] == cell_line) &
                    (stats["Phase"] == phase)
                ]

                if comparison_stats.empty:
                    continue

                stars = comparison_stats.iloc[0][
                    "Significance"
                ]

                if stars:

                    y = (
                        bottom[j]
                        + means[j, i] / 3
                    )

                    ax.text(
                        j,
                        y,
                        stars,
                        ha="center",
                        va="center",
                        fontsize=14
                    )

        bottom += means[:, i]

    # --------------------------------------------------------
    # Formatting
    # --------------------------------------------------------

    ax.set_xticks(x)

    ax.set_xticklabels(
        labels,
        rotation=0
    )

    ax.set_ylabel(
        f"{marker_name}+ cells (%)"
    )

    ax.set_ylim(
        0,
        105
    )

    ax.legend(
        title="Cell cycle"
    )

    fig.tight_layout()

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    fig.savefig(
        output,
        dpi=600,
        bbox_inches="tight"
    )

    plt.close(fig)

    print(
        f"Saved: {output}"
    )

In [23]:
# ============================================================
# Plot treatment vs DMSO
# ============================================================

def plot_treatment_comparison(
    df,
    marker_name="PARP",
    cell_lines=None,
    treatments=None,
    labels=None,
    stats=None,
    output_dir=None
):
    """
    Plot treatment vs matched DMSO.

    For each cell line and treatment, separate figures are
    generated for each Experiment + DMSO concentration combination.

    Experiment is therefore explicitly respected here.
    """

    plt.rcParams["font.family"] = "Arial"

    if cell_lines is None:

        cell_lines = sorted(
            df["modification"]
            .dropna()
            .unique()
        )

    if treatments is None:

        treatments = sorted(
            [
                treatment
                for treatment in df["treatment"].dropna().unique()
                if treatment != "DMSO"
            ]
        )

    if labels is None:

        labels = cell_lines

    if len(labels) != len(cell_lines):

        raise ValueError(
            "labels and cell_lines must have the same length."
        )

    label_dict = dict(
        zip(
            cell_lines,
            labels
        )
    )

    if output_dir is None:

        output_dir = os.path.join(
            OUTPUT_FIGURE_DIR,
            "Treatment_vs_DMSO"
        )

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Positive population
    # --------------------------------------------------------

    plot_df = df[
        df["Population"] == "pos"
    ].copy()

    colors = [
        "steelblue",
        "lightblue",
        "thistle"
    ]

    # --------------------------------------------------------
    # Loop through cell lines
    # --------------------------------------------------------

    for cell_line in cell_lines:

        cell_line_label = label_dict[
            cell_line
        ]

        df_cl = plot_df[
            plot_df["modification"] == cell_line
        ].copy()

        if df_cl.empty:

            print(
                f"Skipping {cell_line}: no data."
            )

            continue

        # ----------------------------------------------------
        # Loop through treatments
        # ----------------------------------------------------

        for treatment in treatments:

            drug_df = df_cl[
                df_cl["treatment"] == treatment
            ].copy()

            if drug_df.empty:

                print(
                    f"Skipping {cell_line} - {treatment}: "
                    "no treatment data."
                )

                continue

            # ------------------------------------------------
            # Find experiment/DMSO combinations
            # ------------------------------------------------

            combinations = (
                drug_df[
                    [
                        "Experiment",
                        "DMSO_conc"
                    ]
                ]
                .drop_duplicates()
            )

            # ------------------------------------------------
            # Each experiment/condition gets its own figure
            # ------------------------------------------------

            for _, combination in combinations.iterrows():

                experiment = combination["Experiment"]
                dmso_conc = combination["DMSO_conc"]

                # --------------------------------------------
                # Treatment
                # --------------------------------------------

                treat = drug_df[
                    (drug_df["Experiment"] == experiment) &
                    (
                        drug_df["DMSO_conc"].apply(
                            lambda x: values_match(
                                x,
                                dmso_conc
                            )
                        )
                    )
                ].copy()

                # --------------------------------------------
                # Matching DMSO
                # --------------------------------------------

                dmso_candidates = df_cl[
                    (df_cl["treatment"] == "DMSO") &
                    (df_cl["Experiment"] == experiment)
                ].copy()

                dmso = dmso_candidates[
                    dmso_candidates["DMSO_conc"].apply(
                        lambda x: values_match(
                            x,
                            dmso_conc
                        )
                    )
                ].copy()

                if treat.empty or dmso.empty:

                    print(
                        f"Skipping {cell_line} - {treatment} "
                        f"(Experiment={experiment}, "
                        f"DMSO={dmso_conc}): "
                        "matching DMSO not found."
                    )

                    continue

                # --------------------------------------------
                # Means and SEM
                # --------------------------------------------

                means = np.array(
                    [
                        [
                            dmso["G1"].mean(),
                            dmso["S"].mean(),
                            dmso["G2"].mean()
                        ],
                        [
                            treat["G1"].mean(),
                            treat["S"].mean(),
                            treat["G2"].mean()
                        ]
                    ]
                )

                errors = np.array(
                    [
                        [
                            dmso["G1"].sem(),
                            dmso["S"].sem(),
                            dmso["G2"].sem()
                        ],
                        [
                            treat["G1"].sem(),
                            treat["S"].sem(),
                            treat["G2"].sem()
                        ]
                    ]
                )

                bar_labels = [
                    "DMSO",
                    treatment
                ]

                # --------------------------------------------
                # Figure
                # --------------------------------------------

                fig_width = 2
                height = 5

                fig, ax = plt.subplots(
                    figsize=(
                        fig_width,
                        height
                    )
                )

                x = np.arange(
                    len(bar_labels)
                )

                bottom = np.zeros(
                    len(bar_labels)
                )

                # --------------------------------------------
                # Plot
                # --------------------------------------------

                for i, phase in enumerate(PHASES):

                    ax.bar(
                        x,
                        means[:, i],
                        bottom=bottom,
                        yerr=errors[:, i],
                        width=0.6,
                        capsize=1.5,
                        color=colors[i],
                        label=phase,
                        error_kw={
                            "elinewidth": 0.6,
                            "capthick": 0.6
                        }
                    )

                    # ----------------------------------------
                    # Significance on treatment bar
                    # ----------------------------------------

                    if stats is not None and not stats.empty:

                        comparison_stats = stats[
                            (stats["Cell_line"] == cell_line) &
                            (stats["Treatment"] == treatment) &
                            (stats["Experiment"] == experiment) &
                            (stats["Phase"] == phase)
                        ]

                        # DMSO concentration may be NaN
                        if not comparison_stats.empty:

                            if pd.isna(dmso_conc):

                                comparison_stats = (
                                    comparison_stats[
                                        comparison_stats[
                                            "DMSO_conc"
                                        ].isna()
                                    ]
                                )

                            else:

                                comparison_stats = (
                                    comparison_stats[
                                        comparison_stats[
                                            "DMSO_conc"
                                        ] == dmso_conc
                                    ]
                                )

                        if not comparison_stats.empty:

                            stars = comparison_stats.iloc[0][
                                "Significance"
                            ]

                            if stars:

                                y = (
                                    bottom[1]
                                    + means[1, i] / 3
                                )

                                ax.text(
                                    1,
                                    y,
                                    stars,
                                    ha="center",
                                    va="center",
                                    fontsize=14
                                )

                    bottom += means[:, i]

                # --------------------------------------------
                # Formatting
                # --------------------------------------------

                ax.set_xticks(x)

                ax.set_xticklabels(
                    bar_labels
                )

                ax.set_ylabel(
                    f"{marker_name}+ cells (%)"
                )

                ax.set_title(
                    cell_line_label
                )

                ax.set_ylim(
                    0,
                    105
                )

                ax.legend(
                    title="Cell cycle"
                )

                fig.tight_layout()

                # --------------------------------------------
                # Filename
                # --------------------------------------------

                safe_marker = sanitize_filename(
                    marker_name
                )

                safe_cell_line = sanitize_filename(
                    cell_line_label
                )

                safe_treatment = sanitize_filename(
                    treatment
                )

                safe_experiment = sanitize_filename(
                    experiment
                )

                safe_dmso = sanitize_filename(
                    dmso_conc
                )

                filename = (
                    f"{safe_marker}_"
                    f"{safe_cell_line}_"
                    f"{safe_treatment}_"
                    f"{safe_experiment}_"
                    f"DMSO{safe_dmso}.pdf"
                )

                output_path = os.path.join(
                    output_dir,
                    filename
                )

                fig.savefig(
                    output_path,
                    dpi=600,
                    bbox_inches="tight"
                )

                plt.close(fig)

                print(
                    f"Saved: {output_path}"
                )

In [24]:
# ============================================================
# Generate genotype figures
# ============================================================

# ------------------------------------------------------------
# PARP — WT vs selected KO cell lines
#
# Change cell_lines and labels here to plot another subset.
# The first cell line is always the reference/WT.
# ------------------------------------------------------------

plot_genotype_comparison(
    df=clean_parp,
    marker_name="PARP",

    cell_lines=[
        "C631",
        "ARID1A KO",
        "ARID1B KO",
        "SMARCC1 KO",
        "SMARCA4 KO"
    ],

    labels=[
        "WT",
        "ARID1A",
        "ARID1B",
        "SMARCC1",
        "SMARCA4"
    ],

    treatment="DMSO",

    stats=parp_genotype_stats,

    output=os.path.join(
        OUTPUT_FIGURE_DIR,
        "PARP_DMSO_cellcycle_distribution.pdf"
    )
)


# ------------------------------------------------------------
# γH2AX — WT vs selected KO cell lines
# ------------------------------------------------------------

plot_genotype_comparison(
    df=clean_yh2ax,
    marker_name="yH2AX",

    cell_lines=[
        "C631",
        "ARID1A KO",
        "ARID1B KO",
        "SMARCA4 KO"
    ],

    labels=[
        "WT",
        "ARID1A",
        "ARID1B",
        "SMARCA4"
    ],

    treatment="DMSO",

    stats=yh2ax_genotype_stats,

    output=os.path.join(
        OUTPUT_FIGURE_DIR,
        "yH2AX_DMSO_cellcycle_distribution.pdf"
    )
)


# ============================================================
# Generate treatment figures
# ============================================================

# ------------------------------------------------------------
# PARP — treatment vs DMSO
#
# Change cell_lines and treatments here to analyze other
# cell lines or compounds.
# ------------------------------------------------------------

plot_treatment_comparison(
    df=clean_parp,
    marker_name="PARP",

    cell_lines=[
        "C631",
        "ARID1A KO",
        "ARID1B KO",
        "SMARCC1 KO",
        "SMARCA4 KO"
    ],

    treatments=[
        "MMS"
    ],

    labels=[
        "WT",
        "ARID1A",
        "ARID1B",
        "SMARCC1",
        "SMARCA4"
    ],

    stats=parp_treatment_stats,

    output_dir=os.path.join(
        OUTPUT_FIGURE_DIR,
        "PARP_Treatment_vs_DMSO"
    )
)


# ------------------------------------------------------------
# γH2AX — treatment vs DMSO
# ------------------------------------------------------------

plot_treatment_comparison(
    df=clean_yh2ax,
    marker_name="yH2AX",

    cell_lines=[
        "C631",
        "ARID1A KO",
        "SMARCA4 KO"
    ],

    treatments=[
        "MMS"
    ],

    labels=[
        "WT",
        "ARID1A",
        "SMARCA4"
    ],

    stats=yh2ax_treatment_stats,

    output_dir=os.path.join(
        OUTPUT_FIGURE_DIR,
        "yH2AX_Treatment_vs_DMSO"
    )
)

Saved: C:/Users/aretakie/Desktop/Extra_Phenotypes/testrun/Figures\PARP_DMSO_cellcycle_distribution.pdf
Saved: C:/Users/aretakie/Desktop/Extra_Phenotypes/testrun/Figures\yH2AX_DMSO_cellcycle_distribution.pdf
Saved: C:/Users/aretakie/Desktop/Extra_Phenotypes/testrun/Figures\PARP_Treatment_vs_DMSO\PARP_WT_MMS_Exp1_DMSO0.03%.pdf
Saved: C:/Users/aretakie/Desktop/Extra_Phenotypes/testrun/Figures\PARP_Treatment_vs_DMSO\PARP_WT_MMS_Exp2_DMSO0.03%.pdf
Saved: C:/Users/aretakie/Desktop/Extra_Phenotypes/testrun/Figures\PARP_Treatment_vs_DMSO\PARP_ARID1A_MMS_Exp1_DMSO0.03%.pdf
Saved: C:/Users/aretakie/Desktop/Extra_Phenotypes/testrun/Figures\PARP_Treatment_vs_DMSO\PARP_ARID1B_MMS_Exp1_DMSO0.03%.pdf
Skipping SMARCC1 KO - MMS: no treatment data.
Saved: C:/Users/aretakie/Desktop/Extra_Phenotypes/testrun/Figures\PARP_Treatment_vs_DMSO\PARP_SMARCA4_MMS_Exp1_DMSO0.03%.pdf
Saved: C:/Users/aretakie/Desktop/Extra_Phenotypes/testrun/Figures\yH2AX_Treatment_vs_DMSO\yH2AX_WT_MMS_Exp1_DMSO0.03%.pdf
Saved: C:/Us

In [25]:
# ============================================================
# End of analysis
# ============================================================

print("\n" + "=" * 60)
print("Extra phenotypes analysis completed.")
print("=" * 60)

print(
    f"\nFigures saved to:\n"
    f"  {OUTPUT_FIGURE_DIR}"
)

print(
    f"\nStatistics saved to:\n"
    f"  {OUTPUT_STATS_DIR}"
)

print(
    f"\nCell-count table saved to:\n"
    f"  {OUTPUT_TABLE_DIR}"
)


Extra phenotypes analysis completed.

Figures saved to:
  C:/Users/aretakie/Desktop/Extra_Phenotypes/testrun/Figures

Statistics saved to:
  C:/Users/aretakie/Desktop/Extra_Phenotypes/testrun/Statistics

Cell-count table saved to:
  C:/Users/aretakie/Desktop/Extra_Phenotypes/testrun
